In [2]:
# =========================
# SCRAPER MEJORADO
# =========================

import os
import re
import time
from datetime import datetime
from pymongo import MongoClient, UpdateOne
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException, StaleElementReferenceException

# -------------------------
# CONFIGURACIÓN GENERAL
# -------------------------
URL = "https://www.serviceshop.cl/llena-tu-despensa"
NOMBRE_GRUPO = "Ave Mayo"
LIMITE_PAGINAS = 15
MONGO_URI = "mongodb://mongodb:27017/"   # usa el mismo host en todo el notebook
DB_NAME = "Canasta_db"
COLLECTION_NAME = "Retail_A"

# -------------------------
# LIMPIEZA INICIAL
# -------------------------
os.system("pkill -9 chrome >/dev/null 2>&1")
os.system("pkill -9 chromedriver >/dev/null 2>&1")
os.system("rm -rf /tmp/.com.google.Chrome.* >/dev/null 2>&1")
os.system("rm -rf /tmp/.org.chromium.Chromium.* >/dev/null 2>&1")
print("🧹 Limpieza de procesos y temporales completada.")

# -------------------------
# FUNCIONES AUXILIARES
# -------------------------
def limpiar_precio(texto):
    if not texto:
        return 0.0
    texto = texto.strip()
    numeros = re.sub(r"[^\d]", "", texto)
    return float(numeros) if numeros else 0.0

def iniciar_driver():
    options = Options()
    options.binary_location = "/usr/bin/google-chrome"
    options.add_argument("--headless=new")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-gpu")
    options.add_argument("--disable-software-rasterizer")
    options.add_argument("--window-size=1920,1080")
    options.add_argument("--remote-debugging-port=9222")
    options.add_argument("--lang=es-CL")
    options.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    )
    driver = webdriver.Chrome(options=options)
    driver.set_page_load_timeout(60)
    return driver

def esperar_productos(driver, timeout=25):
    WebDriverWait(driver, timeout).until(
        EC.presence_of_all_elements_located((By.CSS_SELECTOR, ".product-block"))
    )
    time.sleep(2)

def extraer_productos_pagina(driver, vistos):
    productos = []
    bloques = driver.find_elements(By.CSS_SELECTOR, ".product-block")

    for bloque in bloques:
        try:
            nombre = ""
            precio_texto = ""

            try:
                nombre = bloque.find_element(By.CSS_SELECTOR, "h3").text.strip()
            except:
                pass

            if not nombre:
                try:
                    nombre = bloque.text.split("\n")[0].strip()
                except:
                    pass

            try:
                precio_texto = bloque.find_element(By.CSS_SELECTOR, ".price").text.strip()
            except:
                pass

            if not nombre:
                continue

            clave = f"{nombre}|{precio_texto}"
            if clave in vistos:
                continue
            vistos.add(clave)

            productos.append({
                "identificador": nombre,
                "valor_texto": precio_texto,
                "valor": limpiar_precio(precio_texto),
                "fecha_captura": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                "grupo": NOMBRE_GRUPO,
                "url_origen": driver.current_url
            })

        except StaleElementReferenceException:
            continue
        except Exception:
            continue

    return productos

def intentar_siguiente_pagina(driver):
    posibles_selectores = [
        "a.next",
        "a[rel='next']",
        ".pagination a.next",
        ".woocommerce-pagination .next",
        "li.next a",
        "a.page-numbers.next",
        ".nav-links a.next"
    ]

    for selector in posibles_selectores:
        try:
            botones = driver.find_elements(By.CSS_SELECTOR, selector)
            for boton in botones:
                if boton.is_displayed() and boton.is_enabled():
                    url_anterior = driver.current_url
                    driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", boton)
                    time.sleep(1)
                    driver.execute_script("arguments[0].click();", boton)

                    WebDriverWait(driver, 15).until(
                        lambda d: d.current_url != url_anterior
                    )
                    time.sleep(2)
                    return True
        except:
            continue

    return False

def guardar_mongodb(datos):
    if not datos:
        print("No hay datos para guardar en MongoDB.")
        return

    client = MongoClient(MONGO_URI, serverSelectionTimeoutMS=5000)
    db = client[DB_NAME]
    coleccion = db[COLLECTION_NAME]

    operaciones = []
    for d in datos:
        filtro = {
            "identificador": d["identificador"],
            "grupo": d["grupo"],
            "url_origen": d["url_origen"]
        }
        operaciones.append(
            UpdateOne(
                filtro,
                {"$set": d},
                upsert=True
            )
        )

    if operaciones:
        resultado = coleccion.bulk_write(operaciones)
        print(f"Datos guardados en MongoDB. Upserts: {resultado.upserted_count}, Modificados: {resultado.modified_count}")

# -------------------------
# EJECUCIÓN PRINCIPAL
# -------------------------
driver = None
datos_finales = []
vistos = set()

try:
    driver = iniciar_driver()
    print("✅ Navegador iniciado correctamente.")

    driver.get(URL)
    esperar_productos(driver)

    for pagina in range(1, LIMITE_PAGINAS + 1):
        print(f"--- Procesando página {pagina} ---")

        esperar_productos(driver)
        productos_pagina = extraer_productos_pagina(driver, vistos)
        datos_finales.extend(productos_pagina)

        print(f"Productos extraídos en esta página: {len(productos_pagina)}")
        print(f"Total acumulado: {len(datos_finales)}")

        avanzo = intentar_siguiente_pagina(driver)
        if not avanzo:
            print("No se encontró paginación adicional o ya es la última página.")
            break

    print(f"✅ Extracción terminada: {len(datos_finales)} productos únicos.")

except TimeoutException as e:
    print(f"⏰ Timeout en Selenium: {e}")
except Exception as e:
    print(f"❌ Error en Selenium: {e}")
finally:
    if driver is not None:
        try:
            driver.quit()
        except:
            pass

# -------------------------
# GUARDAR EN MONGODB
# -------------------------
try:
    guardar_mongodb(datos_finales)
except Exception as e:
    print(f"❌ Error en MongoDB: {e}")

🧹 Limpieza de procesos y temporales completada.
✅ Navegador iniciado correctamente.
--- Procesando página 1 ---
Productos extraídos en esta página: 40
Total acumulado: 40
No se encontró paginación adicional o ya es la última página.
✅ Extracción terminada: 40 productos únicos.
Datos guardados en MongoDB. Upserts: 0, Modificados: 40
